# 03 - Construccion de features a nivel de persona

Construye las features analiticas para el analisis posterior de clustering, a partir de
`data/features/dataset_personas_integrado.csv` (1 fila = 1 persona, generado en
`02_integracion_datos.ipynb`).

**Principio:** cada feature construida debe tener una justificacion ligada a una
dimension del perfil profesional (formacion, experiencia, docencia, investigacion,
publicaciones, capacitacion, certificaciones, ponencias, vinculacion, idiomas,
evaluacion docente, reconocimientos, trayectoria laboral, carga politecnica). No se
crean features solo porque una columna existe.

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent.parent
FEATURES_DIR = PROJECT_ROOT / "data" / "features"

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)

## 1. Carga del dataset integrado (solo lectura)

In [ ]:
dataset_personas_integrado = pd.read_csv(FEATURES_DIR / "dataset_personas_integrado.csv")

print("Shape:", dataset_personas_integrado.shape)
print("Numero de personas (IDPERSONA unicos):", dataset_personas_integrado["IDPERSONA"].nunique())
print("Numero de columnas:", dataset_personas_integrado.shape[1])

In [ ]:
def resumen_columnas(df: pd.DataFrame) -> pd.DataFrame:
    """Resumen de tipos, nulos, unicos y deteccion de columnas (casi) constantes."""
    resumen = pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "pct_nulos": (df.isna().mean() * 100).round(2),
        "n_unicos": df.nunique(dropna=True),
    })
    top_share = {}
    for c in df.columns:
        vc = df[c].value_counts(dropna=True, normalize=True)
        top_share[c] = round(vc.iloc[0] * 100, 2) if len(vc) else np.nan
    resumen["pct_valor_dominante"] = pd.Series(top_share)
    resumen["constante"] = resumen["n_unicos"] <= 1
    resumen["casi_constante"] = resumen["pct_valor_dominante"] >= 97
    return resumen

resumen_inicial = resumen_columnas(dataset_personas_integrado)
resumen_inicial

In [ ]:
print("Columnas constantes o casi constantes (>=97% valor dominante):")
resumen_inicial[resumen_inicial["constante"] | resumen_inicial["casi_constante"]]

## 2. Clasificacion de tipos de variable

Clasificacion preliminar  `col_categoricas` y
`col_booleanas`.

In [ ]:
col_identificador = ["IDPERSONA"]

col_fecha = ["FECHANACIMIENTO", "FECHA_PRIMER_INGRESO", "FECHA_ULTIMO_PERIODO_FIN"]

col_booleanas = [
    "MULTIPLES_CARGOS_MISMO_ANIO", "PASO_POR_RECTORADO",
    "ES_DOCENTE_ADMIN_MIXTO", "VIGENTE_ACTUALMENTE",
]

col_categoricas = [
    "ESTADOCIVIL", "SEXO", "PAISORIGEN", "PROVINCIAORIGEN", "CANTONORIGEN",
    "CARGO_ACTUAL", "CARGO_MAS_FRECUENTE", "UNIDAD_ACTUAL_NOMBRE",
    "REGIMEN_INICIAL_DESC", "REGIMEN_ACTUAL_DESC", "TIPOEMPLEADO_ACTUAL_DESC",
    "DEDICACION_DOCENTE_ACTUAL", "DEDICACION_DOCENTE_MAS_FRECUENTE",
]

col_numericas = [
    c for c in dataset_personas_integrado.columns
    if c not in col_identificador + col_fecha + col_booleanas + col_categoricas
]

# no hay columnas de texto libre (descripciones, comentarios) en el dataset integrado
col_texto_libre = []

print(f"identificador={len(col_identificador)} | fecha={len(col_fecha)} | "
      f"booleanas={len(col_booleanas)} | categoricas={len(col_categoricas)} | "
      f"numericas={len(col_numericas)} | texto_libre={len(col_texto_libre)}")

assert set(col_identificador + col_fecha + col_booleanas + col_categoricas + col_numericas) \
    == set(dataset_personas_integrado.columns)

## 3. Nota sobre fecha de corte

No existe una `FECHA_CORTE` explicita registrada en `context/DECISION_LOG.md`. Al inspeccionar
`FECHA_ULTIMO_PERIODO_FIN` del dataset integrado se observa que, para personas vigentes, el
valor coincide con la fecha de ejecucion del preprocesamiento (usada como placeholder de "hoy"),
y para algunas personas no vigentes aparecen fechas administrativas futuras (ej. fin de
contrato formal). Esto hace que las fechas crudas de `historial_laboral_features.csv` no sean
un indicador confiable de recencia por si mismas.

**Decision de este notebook:** no se inventa una fecha de corte nueva. Se usan las variables ya
resumidas rio arriba (`ANTIGUEDAD_EFECTIVA_ANIOS`, `ANTIGUEDAD_CALENDARIO_ANIOS`,
`VIGENTE_ACTUALMENTE`) como señal temporal, y se excluyen las fechas crudas
(`FECHA_PRIMER_INGRESO`, `FECHA_ULTIMO_PERIODO_FIN`) del dataset de features (ver seccion 4).
Se recomienda registrar una `FECHA_CORTE` explicita en `context/DECISION_LOG.md` para que
`01`/`02` la usen de forma consistente en futuras ejecuciones.

## 4. Tabla de exclusion

Variables que no representan una dimension del perfil profesional, o que son redundantes
(duplicados deterministas o casi deterministas) o casi constantes. No se eliminan
automaticamente: se documenta la razon de cada una antes de aplicarla.

In [ ]:
exclusiones = [
    ("ESTADOCIVIL", "Dato demografico/personal, no representa una dimension profesional del perfil", "ELIMINAR"),
    ("FECHANACIMIENTO", "Fecha personal identificable; EDAD ya deriva de esta y tambien se excluye", "ELIMINAR"),
    ("SEXO", "Dato demografico; no es una dimension profesional y puede introducir sesgo en el perfilamiento", "ELIMINAR"),
    ("PAISORIGEN", "Dato demografico/origen, no representa trayectoria profesional", "ELIMINAR"),
    ("PROVINCIAORIGEN", "Dato demografico/origen, no representa trayectoria profesional", "ELIMINAR"),
    ("CANTONORIGEN", "Dato demografico/origen, alta cardinalidad, no representa trayectoria profesional", "ELIMINAR"),
    ("EDAD", "Dato demografico derivado de FECHANACIMIENTO; ANTIGUEDAD_EFECTIVA_ANIOS ya cubre la nocion temporal profesional relevante", "ELIMINAR"),
    ("NUM_TIT_PRIMARIA", "Casi constante (98.4% en 0); bajo valor informativo como columna independiente. Se usa unicamente como insumo de NIVEL_ACADEMICO_MAXIMO", "ELIMINAR"),
    ("TIENE_POSGRADO", "Duplicado determinista de NUM_TIT_CUARTO_NIVEL > 0 (correlacion=1.0, correspondencia exacta verificada)", "ELIMINAR"),
    ("NUM_ASIGNACIONES_DOCENCIA", "Duplicado casi exacto de NUM_CURSOS (correlacion=1.0)", "ELIMINAR"),
    ("FECHA_PRIMER_INGRESO", "Fecha cruda; informacion resumida en ANTIGUEDAD_EFECTIVA_ANIOS/ANTIGUEDAD_CALENDARIO_ANIOS", "ELIMINAR"),
    ("FECHA_ULTIMO_PERIODO_FIN", "Fecha cruda con valores placeholder (ver nota de fecha de corte); no confiable como indicador de recencia", "ELIMINAR"),
    ("N_PERIODOS_CONTINUOS", "Duplicado determinista de N_REINGRESOS (=N_REINGRESOS + 1, correlacion=1.0); se conserva N_REINGRESOS por ser mas interpretable", "ELIMINAR"),
    ("ANTIGUEDAD_EFECTIVA_DIAS", "Duplicado de ANTIGUEDAD_EFECTIVA_ANIOS en otra unidad (correlacion=1.0)", "ELIMINAR"),
    ("ANTIGUEDAD_CALENDARIO_DIAS", "Duplicado de ANTIGUEDAD_CALENDARIO_ANIOS en otra unidad (correlacion=1.0)", "ELIMINAR"),
]
tabla_exclusion = pd.DataFrame(exclusiones, columns=["COLUMN_NAME", "REASON", "ACTION"])
tabla_exclusion

In [ ]:
# NUM_VINCULACION_DIRECTOR_PROGRAMA tambien es casi constante (99.2% en 0) pero se conserva:
# representa un rol distintivo minoritario (direccion de programas de vinculacion) relevante
# para diferenciar perfiles de alta actividad de vinculacion (ver seccion 18 del brief).

cols_excluidas = tabla_exclusion["COLUMN_NAME"].tolist()
print(f"Columnas a excluir: {len(cols_excluidas)} de {dataset_personas_integrado.shape[1]}")

## 5. Features derivadas

Se construyen unicamente features con justificacion clara. Antes de cada una se verifica que
las columnas de origen existan.

**No disponibles en el dataset integrado (se documenta, no se inventan):**
- Formacion: fecha de titulacion (no hay fecha por titulo) -> no se puede calcular "anios desde
  la ultima graduacion"; area de formacion (no hay columna de area/disciplina) -> no se puede
  calcular diversidad de areas.
- Capacitacion: area de capacitacion -> no se puede calcular diversidad de areas.
- Certificaciones: vigencia -> no hay columna de fecha de expiracion/estado.
- Idiomas: nivel MCER por idioma -> `idiomas_persona.csv` solo trae conteos agregados
  (`NUM_IDIOMAS`, `NUM_IDIOMAS_NO_NATIVOS`, `NUM_IDIOMAS_CON_NIVELMCER`), no el nivel en si;
  no se puede construir un maximo MCER ni una escala ordinal A1-C2.

In [ ]:
# --- Formacion: nivel academico maximo ---
# Se calcula ANTES de excluir NUM_TIT_PRIMARIA (esta se usa como insumo de la jerarquia
# aunque no se conserve como columna independiente, ver seccion 4).
# Jerarquia estandar de niveles educativos (verificada con conteos > 0, no supuesta a ciegas):
orden_nivel = ["sin_registro", "primaria", "bachillerato", "tercer_nivel", "cuarto_nivel"]

def nivel_academico_maximo(row):
    if row["NUM_TIT_CUARTO_NIVEL"] > 0:
        return "cuarto_nivel"
    if row["NUM_TIT_TERCER_NIVEL"] > 0:
        return "tercer_nivel"
    if row["NUM_TIT_BACHILLERATO"] > 0:
        return "bachillerato"
    if row["NUM_TIT_PRIMARIA"] > 0:
        return "primaria"
    return "sin_registro"

nivel_academico = pd.Categorical(
    dataset_personas_integrado.apply(nivel_academico_maximo, axis=1),
    categories=orden_nivel, ordered=True,
)
pd.Series(nivel_academico).value_counts()

In [ ]:
# --- Construccion del dataframe de trabajo tras exclusiones ---
df = dataset_personas_integrado.drop(columns=cols_excluidas).copy()
df["NIVEL_ACADEMICO_MAXIMO"] = nivel_academico
print("Shape tras exclusion + NIVEL_ACADEMICO_MAXIMO:", df.shape)

In [ ]:
# --- Booleanas a tipo nullable boolean (se preserva NaN, no se fuerza a 0/1) ---
for c in col_booleanas:
    df[c] = df[c].astype("boolean")
df[col_booleanas].dtypes

In [ ]:
# --- Capacitacion: proporcion de capacitaciones aprobadas ---
# Solo se calcula si hubo capacitaciones; si NUM_CAPACITACIONES=0 el resultado es NaN
# (no aplica, no es 0% de aprobacion).
df["PROPORCION_CAPACITACIONES_APROBADAS"] = np.where(
    df["NUM_CAPACITACIONES"] > 0,
    df["NUM_CAPACITACIONES_APROBACION"] / df["NUM_CAPACITACIONES"],
    np.nan,
)
df["PROPORCION_CAPACITACIONES_APROBADAS"].describe()

In [ ]:
# --- Carga politecnica: horas promedio por actividad (intensidad) ---
df["HORAS_PROMEDIO_POR_ACTIVIDAD_POLITECNICA"] = np.where(
    df["NUM_ACTIVIDADES_POLITECNICAS"] > 0,
    df["TOTAL_HORAS_POLITECNICAS"] / df["NUM_ACTIVIDADES_POLITECNICAS"],
    np.nan,
)
df["HORAS_PROMEDIO_POR_ACTIVIDAD_POLITECNICA"].describe()

In [ ]:
# --- Investigacion / publicaciones: tasa anual normalizada por exposicion ---
# Se usa ANTIGUEDAD_EFECTIVA_ANIOS >= 1 como guarda: para antiguedades menores a un anio
# la tasa es inestable (denominador cercano a 0) y se deja NaN en vez de un valor extremo.
antig_valida = df["ANTIGUEDAD_EFECTIVA_ANIOS"] >= 1
print("Personas con antiguedad efectiva < 1 anio o nula (tasas quedan NaN):", (~antig_valida).sum())

df["NUM_PUBLICACIONES_POR_ANIO"] = np.where(
    antig_valida, df["NUM_PUBLICACIONES"] / df["ANTIGUEDAD_EFECTIVA_ANIOS"], np.nan)
df["NUM_PROYECTOS_INVESTIGACION_POR_ANIO"] = np.where(
    antig_valida, df["NUM_PROYECTOS_INVESTIGACION"] / df["ANTIGUEDAD_EFECTIVA_ANIOS"], np.nan)

df[["NUM_PUBLICACIONES_POR_ANIO", "NUM_PROYECTOS_INVESTIGACION_POR_ANIO"]].describe()

## 6. Ceros vs nulos

Regla aplicada de forma consistente en todo el notebook: **0 = actividad registrada en cero**
(la persona existe en la fuente pero no tiene ocurrencias), **NaN = informacion no disponible**
(no se puede calcular o la fuente no cubre a esa persona). No se convierten NaN a 0 en ningun
punto de este notebook.

Casos revisados explicitamente:
- `PROMEDIO_ESTUDIANTES_POR_CURSO`, `PROMEDIO_HETEROEVALUACION`, `COBERTURA_EVALUACION`: NaN
  cuando el denominador (cursos/evaluaciones) es 0 -> correctamente NaN, ya calculado en `02`.
- `PROPORCION_CAPACITACIONES_APROBADAS`, `HORAS_PROMEDIO_POR_ACTIVIDAD_POLITECNICA`,
  `NUM_PUBLICACIONES_POR_ANIO`, `NUM_PROYECTOS_INVESTIGACION_POR_ANIO`: mismo criterio,
  aplicado arriba con `np.where`.
- Las columnas de trayectoria laboral (`N_REGISTROS_HISTORIAL`, `N_CONTRATOS_TOTAL`, etc.)
  tienen ~0.2% de nulos: personas sin registro en `historial_laboral_features.csv`. Se
  mantienen como NaN (dato desconocido), no se imputan a 0, porque no hay evidencia de que
  la ausencia signifique "cero contratos" para poblacion de personal contratado.
- `REGIMEN_INICIAL_DESC` (24.3% nulo) y `DEDICACION_DOCENTE_*` (~24.9% nulo): nulo se
  mantiene como categoria "informacion no disponible" (no se imputa una categoria).

## 7. Variables categoricas y texto (sin codificar)

In [ ]:
print("Categoricas conservadas (se codifican en 04_preparacion_modelado.ipynb):")
for c in col_categoricas:
    if c in df.columns:
        print(f"  {c}: {df[c].nunique(dropna=True)} categorias, {df[c].isna().mean()*100:.1f}% nulos")

print()
print("CARGO_ACTUAL y CARGO_MAS_FRECUENTE tienen alta cardinalidad (>200 categorias):")
print("  se conservan como etiquetas categoricas del perfil (cargo institucional), pero")
print("  quedan senaladas para posible agrupacion antes de codificar en 04.")
print()
print("No se identificaron columnas de texto libre (descripciones/comentarios) en el dataset integrado.")

## 8. Redundancia entre variables numericas

Diagnostico de correlacion sobre el conjunto YA depurado (tras la seccion 4). Los duplicados
deterministas o casi deterministas ya fueron eliminados alli con su propia justificacion; lo
que queda aqui son pares correlacionados pero conceptualmente distintos (parte-todo,
exposicion compartida), que se documentan y se conservan. La seleccion final de variables se
hace en el siguiente notebook.

In [ ]:
num_cols = df.select_dtypes(include=[np.number]).columns.drop("IDPERSONA")
corr = df[num_cols].corr()

JUSTIFICACIONES_CORR = {
    frozenset(["TOTAL_REGISTRADOS", "TOTAL_EVALUADOS"]): "Numerador y denominador de COBERTURA_EVALUACION; se conservan ambos por transparencia, la ratio ya resume su relacion.",
    frozenset(["NUM_PUBLICACIONES", "NUM_PUBLICACIONES_INDEXADAS"]): "Total vs subconjunto indexado: distincion de calidad relevante para el perfil de investigacion.",
    frozenset(["NUM_CURSOS", "NUM_EVALUACIONES"]): "Mas cursos generan mas evaluaciones, pero representan conceptos distintos (docencia vs evaluacion docente).",
    frozenset(["ANTIGUEDAD_EFECTIVA_ANIOS", "ANTIGUEDAD_CALENDARIO_ANIOS"]): "Antiguedad efectiva (activa) vs calendario (incluye interrupciones); distincion relevante para reingresos.",
    frozenset(["TOTAL_HORAS_DOCENCIA", "TOTAL_ESTUDIANTES"]): "Horas de docencia vs volumen de estudiantes: conceptos distintos de exposicion docente.",
    frozenset(["NUM_CURSOS", "TOTAL_HORAS_DOCENCIA"]): "Numero de cursos vs horas totales: cursos pueden diferir en duracion.",
    frozenset(["NUM_PROYECTOS_INVESTIGACION", "NUM_PROYECTOS_COMO_PARTICIPANTE"]): "Total de proyectos vs desagregacion por rol (participante); se conserva el detalle por rol.",
    frozenset(["NUM_PROYECTOS_INVESTIGACION", "NUM_PROYECTOS_FINALIZADOS"]): "Total de proyectos vs desagregacion por estado (finalizado); se conserva el detalle de estado.",
    frozenset(["NUM_PROYECTOS_INVESTIGACION", "NUM_PROYECTOS_ACTIVOS"]): "Total de proyectos vs desagregacion por estado (activo); se conserva el detalle de estado.",
    frozenset(["NUM_CURSOS", "TOTAL_ESTUDIANTES"]): "Numero de cursos vs volumen de estudiantes atendidos.",
    frozenset(["NUM_PERIODOS_DOCENCIA", "TOTAL_HORAS_DOCENCIA"]): "Periodos con docencia vs horas totales acumuladas.",
    frozenset(["NUM_PROYECTOS_ACTIVOS", "NUM_PROYECTOS_COMO_PARTICIPANTE"]): "Estado (activo) vs rol (participante) de proyectos; dimensiones distintas.",
}
DEFAULT_JUST = ("Correlacion esperada por relacion parte-todo o por exposicion compartida; "
                 "se conservan ambas variables porque aportan granularidad distinta "
                 "(conteo total vs desagregacion por termino/tipo).")

pares = []
cols = corr.columns
for i in range(len(cols)):
    for j in range(i + 1, len(cols)):
        v = corr.iloc[i, j]
        if pd.notna(v) and abs(v) >= 0.85:
            key = frozenset([cols[i], cols[j]])
            just = JUSTIFICACIONES_CORR.get(key, DEFAULT_JUST)
            pares.append((cols[i], cols[j], round(v, 3), "CONSERVAR", just))
pares.sort(key=lambda x: -abs(x[2]))

feature_correlations = pd.DataFrame(
    pares, columns=["VARIABLE_1", "VARIABLE_2", "CORRELACION", "DECISION", "JUSTIFICACION"])
print(f"{len(feature_correlations)} pares con |correlacion| >= 0.85 (post-limpieza)")
feature_correlations

## 9. Distribuciones numericas

In [ ]:
def diagnostico_numericas(df, cols):
    """Estadisticas descriptivas, percentiles, asimetria y % de ceros por variable."""
    filas = []
    for c in cols:
        s = df[c]
        filas.append({
            "FEATURE": c, "N_NULL": int(s.isna().sum()), "PCT_NULL": round(s.isna().mean() * 100, 2),
            "PCT_CEROS": round((s == 0).sum() / s.notna().sum() * 100, 2) if s.notna().any() else np.nan,
            "MIN": s.min(), "P25": s.quantile(.25), "P50": s.quantile(.5), "P75": s.quantile(.75),
            "P95": s.quantile(.95), "MAX": s.max(), "MEAN": round(s.mean(), 3), "STD": round(s.std(), 3),
            "SKEW": round(s.skew(), 3),
        })
    return pd.DataFrame(filas)

feature_diagnostics = diagnostico_numericas(df, num_cols.tolist())
feature_diagnostics.sort_values("SKEW", ascending=False).head(15)

In [ ]:
# Variables muy sesgadas (|skew|>=2): candidatas a log1p en 04_preparacion_modelado.ipynb.
# No se aplica ninguna transformacion aqui.
sesgadas = feature_diagnostics[feature_diagnostics["SKEW"].abs() >= 2].sort_values("SKEW", ascending=False)
print(f"{len(sesgadas)} variables con |skew| >= 2 (de {len(feature_diagnostics)} numericas)")
sesgadas[["FEATURE", "SKEW", "PCT_CEROS", "MAX"]]

In [ ]:
# Valores infinitos (no deberian existir; se verifica antes de continuar)
inf_por_columna = np.isinf(df[num_cols]).sum()
inf_por_columna = inf_por_columna[inf_por_columna > 0]
print("Columnas con valores infinitos:", dict(inf_por_columna) if len(inf_por_columna) else "ninguna")
assert not np.isinf(df[num_cols]).to_numpy().any()

## 10. Ensamblado del dataset final de features

In [ ]:
df = df[["IDPERSONA"] + [c for c in df.columns if c != "IDPERSONA"]]
dataset_personas_features = df
print("Shape final:", dataset_personas_features.shape)
dataset_personas_features.head(3)

## 11. Validaciones finales

In [ ]:
n_antes = dataset_personas_integrado["IDPERSONA"].nunique()
n_despues = dataset_personas_features["IDPERSONA"].nunique()

print("IDPERSONA unico:", dataset_personas_features["IDPERSONA"].is_unique)
print(f"Personas antes: {n_antes} | despues: {n_despues}")
print("Filas duplicadas completas:", dataset_personas_features.duplicated().sum())
print("Numero de features (sin IDPERSONA):", dataset_personas_features.shape[1] - 1)
assert dataset_personas_features["IDPERSONA"].is_unique
assert n_antes == n_despues
assert dataset_personas_features.duplicated().sum() == 0

In [ ]:
pct_nulos_final = (dataset_personas_features.isna().mean() * 100).round(2).sort_values(ascending=False)
print("Top-10 variables con mas nulos en el dataset final:")
pct_nulos_final.head(10)

In [ ]:
constantes_final = [c for c in dataset_personas_features.columns
                     if c != "IDPERSONA" and dataset_personas_features[c].nunique(dropna=True) <= 1]
casi_constantes_final = []
for c in dataset_personas_features.columns:
    if c == "IDPERSONA":
        continue
    vc = dataset_personas_features[c].value_counts(dropna=True, normalize=True)
    if len(vc) and vc.iloc[0] >= 0.97:
        casi_constantes_final.append((c, round(vc.iloc[0] * 100, 1)))

print("Columnas constantes en el dataset final:", constantes_final)
print("Columnas casi constantes en el dataset final:", casi_constantes_final)
assert len(constantes_final) == 0

In [ ]:
n_numericas_final = dataset_personas_features.select_dtypes(include=[np.number]).shape[1] - 1
n_booleanas_final = len(col_booleanas)
n_categoricas_final = sum(1 for c in col_categoricas if c in dataset_personas_features.columns)
n_categoricas_final += 1  # NIVEL_ACADEMICO_MAXIMO

print(f"Numericas: {n_numericas_final} | Booleanas: {n_booleanas_final} | "
      f"Categoricas (incl. NIVEL_ACADEMICO_MAXIMO): {n_categoricas_final}")

In [ ]:
# Rangos imposibles: proporciones/coberturas fuera de [0,1]
prop_cols = [c for c in dataset_personas_features.columns
             if c.startswith("PROPORCION_") or c == "COBERTURA_EVALUACION"]
for c in prop_cols:
    v = dataset_personas_features[c].dropna()
    fuera_rango = v[(v < 0) | (v > 1)]
    print(f"{c}: {len(fuera_rango)} valores fuera de [0,1]")
    assert len(fuera_rango) == 0

# Conteos negativos
count_cols = [c for c in num_cols if c.startswith(("NUM_", "N_", "TOTAL_"))]
negativos = {c: int((dataset_personas_features[c] < 0).sum())
             for c in count_cols if (dataset_personas_features[c] < 0).any()}
print("Columnas de conteo con valores negativos:", negativos if negativos else "ninguna")
assert len(negativos) == 0

In [ ]:
# Correlaciones potencialmente problematicas en el dataset final (>=0.98, casi duplicados
# que se hayan podido pasar por alto)
corr_final = dataset_personas_features[num_cols].corr()
alerta = []
cols_f = corr_final.columns
for i in range(len(cols_f)):
    for j in range(i + 1, len(cols_f)):
        v = corr_final.iloc[i, j]
        if pd.notna(v) and abs(v) >= 0.98:
            alerta.append((cols_f[i], cols_f[j], round(v, 4)))
print("Pares con correlacion >= 0.98 (revisar si son duplicados no detectados):")
alerta

## 12. Archivos auxiliares: diccionario de features

In [ ]:
DIMENSIONES = [
    (["NUM_TITULACIONES", "NUM_TIT_", "NIVEL_ACADEMICO_MAXIMO"], "formacion_academica",
     "Describe el nivel y volumen de titulaciones academicas obtenidas."),
    (["CAPACITACION"], "capacitacion", "Describe volumen, duracion y aprobacion de capacitaciones recibidas."),
    (["CERTIFICACION"], "certificaciones", "Describe volumen y duracion de certificaciones obtenidas."),
    (["PONENCIA"], "ponencias", "Describe participacion como ponente en eventos academicos."),
    (["CURSOS", "PERIODOS_DOCENCIA", "HORAS_DOCENCIA", "ESTUDIANTES"], "docencia",
     "Describe volumen y alcance de la actividad de docencia."),
    (["ACTIVIDADES_POLITECNICAS", "ACTIVIDADES_T1", "ACTIVIDADES_T2", "ACTIVIDADES_T3",
      "HORAS_POLITECNICAS", "HORAS_PROMEDIO_POR_ACTIVIDAD_POLITECNICA"], "carga_politecnica",
     "Describe volumen, distribucion por termino e intensidad de la carga politecnica."),
    (["EXPERIENCIAS_"], "experiencia_externa", "Describe experiencia laboral externa reportada, por tipo."),
    (["IDIOMAS"], "idiomas", "Describe el numero de idiomas registrados y su cobertura de nivel MCER."),
    (["EVALUACIONES", "HETEROEVALUACION", "REGISTRADOS", "EVALUADOS", "COBERTURA_EVALUACION"], "evaluacion_docente",
     "Describe resultados y cobertura de la heteroevaluacion docente."),
    (["RECONOCIMIENTO"], "reconocimientos", "Describe menciones y reconocimientos institucionales recibidos."),
    (["PROYECTOS_GRADO", "PROGRAMAS_TITULACION_DISTINTOS"], "proyectos_grado",
     "Describe direccion de trabajos de titulacion de estudiantes."),
    (["PROYECTOS_INVESTIGACION", "PROYECTOS_ACTIVOS", "PROYECTOS_FINALIZADOS",
      "PROYECTOS_COMO_DIRECTOR", "PROYECTOS_COMO_CODIRECTOR", "PROYECTOS_COMO_PARTICIPANTE",
      "PROYECTOS_INVESTIGACION_POR_ANIO"], "investigacion",
     "Describe volumen, estado y rol de participacion en proyectos de investigacion."),
    (["PROYECTOS_VINCULACION", "VINCULACION_TUTOR", "VINCULACION_DIRECTOR"], "vinculacion",
     "Describe volumen y rol de participacion en proyectos de vinculacion."),
    (["PUBLICACIONES"], "publicaciones",
     "Describe volumen y calidad (indexacion, revision por pares, cuartil) de publicaciones."),
    (["CARGO_", "UNIDAD_ACTUAL", "REGIMEN_", "TIPOEMPLEADO_", "DEDICACION_DOCENTE",
      "N_REGISTROS_HISTORIAL", "N_CONTRATOS_TOTAL", "N_CARGOS_DISTINTOS", "MULTIPLES_CARGOS_MISMO_ANIO",
      "N_UNIDADES_DISTINTAS", "N_FACULTADES_DISTINTAS", "PASO_POR_RECTORADO", "N_REGIMENES_DISTINTOS",
      "ES_DOCENTE_ADMIN_MIXTO", "ANIOS_EXPERIENCIA_", "N_DEDICACIONES_DOCENTE_DISTINTAS",
      "PROPORCION_CONTRATOS_FINALIZADOS", "VIGENTE_ACTUALMENTE", "ANTIGUEDAD_", "N_REINGRESOS"],
     "trayectoria_laboral", "Describe la trayectoria laboral institucional (historial_laboral_features.csv), ya procesada previamente."),
]

def clasificar_dimension(col):
    for keys, source, rationale in DIMENSIONES:
        if any(k in col for k in keys):
            return source, rationale
    return "otro", "Sin clasificar."

def tipo_de(col):
    if col == "IDPERSONA":
        return "identificador"
    if col in col_booleanas:
        return "booleana"
    if col == "NIVEL_ACADEMICO_MAXIMO":
        return "categorica_ordinal"
    if str(dataset_personas_features[col].dtype) in ("object",) or str(dataset_personas_features[col].dtype).startswith("str"):
        return "categorica"
    return "numerica"

DESCRIPCIONES_ESPECIALES = {
    "IDPERSONA": "Identificador unico de persona.",
    "NIVEL_ACADEMICO_MAXIMO": "Nivel academico mas alto alcanzado, derivado de NUM_TIT_* (jerarquia: sin_registro < primaria < bachillerato < tercer_nivel < cuarto_nivel).",
    "PROPORCION_CAPACITACIONES_APROBADAS": "Proporcion de capacitaciones aprobadas sobre el total tomado (NaN si NUM_CAPACITACIONES=0).",
    "HORAS_PROMEDIO_POR_ACTIVIDAD_POLITECNICA": "Horas promedio por actividad de carga politecnica (NaN si no hay actividades).",
    "NUM_PUBLICACIONES_POR_ANIO": "Tasa de publicaciones por anio de antiguedad efectiva (NaN si ANTIGUEDAD_EFECTIVA_ANIOS < 1).",
    "NUM_PROYECTOS_INVESTIGACION_POR_ANIO": "Tasa de proyectos de investigacion por anio de antiguedad efectiva (NaN si ANTIGUEDAD_EFECTIVA_ANIOS < 1).",
}

def descripcion_de(col):
    return DESCRIPCIONES_ESPECIALES.get(col, col.replace("_", " ").lower().capitalize() + ".")

filas_dict = []
for col in dataset_personas_features.columns:
    if col == "IDPERSONA":
        source, rationale = "identificacion", "Llave de union con el resto de tablas del proyecto."
    else:
        source, rationale = clasificar_dimension(col)
    filas_dict.append({
        "FEATURE": col, "SOURCE": source, "TYPE": tipo_de(col),
        "DESCRIPTION": descripcion_de(col), "RATIONALE": rationale,
    })
feature_dictionary = pd.DataFrame(filas_dict)
print("Features sin clasificar (deberia ser 0):", (feature_dictionary["SOURCE"] == "otro").sum())
feature_dictionary["SOURCE"].value_counts()

## 13. Guardar archivos de salida

In [ ]:
features_excluded = tabla_exclusion.rename(columns={"ACTION": "DECISION"})[["COLUMN_NAME", "REASON", "DECISION"]]

dataset_personas_features.to_csv(FEATURES_DIR / "dataset_personas_features.csv", index=False)
feature_dictionary.to_csv(FEATURES_DIR / "feature_dictionary.csv", index=False)
features_excluded.to_csv(FEATURES_DIR / "features_excluded.csv", index=False)
feature_diagnostics.to_csv(FEATURES_DIR / "feature_diagnostics.csv", index=False)
feature_correlations.to_csv(FEATURES_DIR / "feature_correlations.csv", index=False)

print("Archivos guardados en", FEATURES_DIR)
for nombre in ["dataset_personas_features.csv", "feature_dictionary.csv", "features_excluded.csv",
               "feature_diagnostics.csv", "feature_correlations.csv"]:
    print(" -", nombre)

## 14. Resumen final

In [ ]:
features_originales_conservadas = [
    c for c in dataset_personas_integrado.columns
    if c in dataset_personas_features.columns
]
features_nuevas = [
    "NIVEL_ACADEMICO_MAXIMO", "PROPORCION_CAPACITACIONES_APROBADAS",
    "HORAS_PROMEDIO_POR_ACTIVIDAD_POLITECNICA", "NUM_PUBLICACIONES_POR_ANIO",
    "NUM_PROYECTOS_INVESTIGACION_POR_ANIO",
]

print("=" * 60)
print("RESUMEN - 03_construccion_features")
print("=" * 60)
print(f"Dataset origen : {dataset_personas_integrado.shape}")
print(f"Dataset final  : {dataset_personas_features.shape}")
print(f"Personas       : {dataset_personas_features['IDPERSONA'].nunique()}")
print(f"Features totales (sin IDPERSONA): {dataset_personas_features.shape[1] - 1}")
print()
print(f"Features originales conservadas: {len(features_originales_conservadas) - 1}")  # sin IDPERSONA
print(f"Features nuevas (derivadas)    : {len(features_nuevas)}")
print(f"Features eliminadas            : {len(cols_excluidas)}")
print()
print("Razones principales de eliminacion:")
print("  - Demografico/privacidad, no es dimension profesional (7): ESTADOCIVIL, FECHANACIMIENTO,")
print("    SEXO, PAISORIGEN, PROVINCIAORIGEN, CANTONORIGEN, EDAD")
print("  - Duplicado determinista o de unidad (5): TIENE_POSGRADO, NUM_ASIGNACIONES_DOCENCIA,")
print("    N_PERIODOS_CONTINUOS, ANTIGUEDAD_EFECTIVA_DIAS, ANTIGUEDAD_CALENDARIO_DIAS")
print("  - Casi constante (1): NUM_TIT_PRIMARIA")
print("  - Fecha cruda no confiable (2): FECHA_PRIMER_INGRESO, FECHA_ULTIMO_PERIODO_FIN")
print()
print("Features nuevas creadas:")
for f in features_nuevas:
    print(f"  - {f}")

## 15. Visualizaciones de apoyo

Graficos descriptivos del dataset final de features, uno por dimension del perfil, para
verificar visualmente lo ya cuantificado en las secciones anteriores (distribuciones,
nulos, asimetria y redundancia). No se toma ninguna decision nueva aqui: son un
complemento visual de los diagnosticos ya calculados.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

# Misma paleta y funciones reutilizables que 02_integracion_datos.ipynb, para mantener
# consistencia visual entre notebooks del proyecto.
COLOR_BLUE, COLOR_ORANGE, COLOR_AQUA, COLOR_YELLOW = "#2a78d6", "#eb6834", "#1baf7a", "#eda100"
PALETA_CATEGORICA = [COLOR_BLUE, COLOR_ORANGE, COLOR_AQUA, COLOR_YELLOW, "#e87ba4", "#008300", "#4a3aa7", "#e34948"]
INK, MUTED, GRID = "#0b0b0b", "#898781", "#e1e0d9"

plt.rcParams.update({
    "figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb",
    "text.color": INK, "axes.labelcolor": INK,
    "xtick.color": MUTED, "ytick.color": MUTED, "font.size": 10,
})


def _estilizar_ejes(ax):
    """Quita bordes sobrantes y deja los ejes en tono recesivo."""
    for spine in ("top", "right", "left"):
        ax.spines[spine].set_visible(False)
    ax.spines["bottom"].set_color("#c3c2b7")
    ax.set_axisbelow(True)


def grafico_hist(serie, titulo, xlabel, color=COLOR_BLUE, bins=25):
    """Histograma simple para una variable numerica continua."""
    fig, ax = plt.subplots(figsize=(7, 4))
    datos = serie.dropna()
    ax.hist(datos, bins=bins, color=color, edgecolor="#fcfcfb", linewidth=0.6, zorder=3)
    ax.set_xlabel(xlabel); ax.set_ylabel("N. de personas")
    ax.set_title(titulo, loc="left", fontsize=12, pad=10)
    ax.grid(axis="y", color=GRID, linewidth=0.8, zorder=0)
    _estilizar_ejes(ax)
    plt.tight_layout()
    plt.show()


def grafico_barras_categoria(serie, titulo, orden=None, colores=PALETA_CATEGORICA):
    """Barras verticales de conteo para una variable categorica (orden de color fijo)."""
    conteo = serie.dropna().value_counts()
    if orden is not None:
        conteo = conteo.reindex(orden).dropna()
    fig, ax = plt.subplots(figsize=(max(4, 1.2 * len(conteo)), 4))
    ax.bar(conteo.index.astype(str), conteo.values, color=colores[: len(conteo)], zorder=3)
    ax.set_ylabel("N. de personas")
    ax.set_title(titulo, loc="left", fontsize=12, pad=10)
    ax.grid(axis="y", color=GRID, linewidth=0.8, zorder=0)
    _estilizar_ejes(ax)
    for i, v in enumerate(conteo.values):
        ax.text(i, v + max(conteo.values) * 0.015, f"{v}", ha="center", fontsize=9)
    plt.tight_layout()
    plt.show()


def grafico_barh_valor(labels, valores, titulo, xlabel, color=COLOR_BLUE, fmt="{:.0f}%"):
    """Barras horizontales para un indicador de magnitud por variable (nulos, skew, etc.)."""
    fig, ax = plt.subplots(figsize=(8, max(2.5, 0.35 * len(labels))))
    y = range(len(labels))
    ax.barh(y, valores, color=color, height=0.6, zorder=3)
    ax.set_yticks(list(y)); ax.set_yticklabels(labels)
    ax.invert_yaxis()
    ax.set_xlabel(xlabel)
    ax.set_title(titulo, loc="left", fontsize=12, pad=10)
    ax.grid(axis="x", color=GRID, linewidth=0.8, zorder=0)
    _estilizar_ejes(ax)
    rango = max(valores) - min(min(valores), 0)
    for i, v in enumerate(valores):
        ax.text(v + rango * 0.015, i, fmt.format(v), va="center", fontsize=9)
    plt.tight_layout()
    plt.show()

### 15.1 Distribucion de variables numericas clave (una por dimension)

In [ ]:
variables_clave = [
    ("NUM_TITULACIONES", "Formacion: numero de titulaciones", "N. de titulaciones"),
    ("NUM_CAPACITACIONES", "Capacitacion: numero de capacitaciones", "N. de capacitaciones"),
    ("TOTAL_HORAS_DOCENCIA", "Docencia: horas totales de docencia", "Horas de docencia"),
    ("TOTAL_HORAS_POLITECNICAS", "Carga politecnica: horas totales", "Horas de carga politecnica"),
    ("NUM_PUBLICACIONES_POR_ANIO", "Investigacion/publicaciones: tasa anual de publicaciones", "Publicaciones por anio de antiguedad"),
    ("ANTIGUEDAD_EFECTIVA_ANIOS", "Trayectoria laboral: antiguedad efectiva", "Anios de antiguedad efectiva"),
]
for col, titulo, xlabel in variables_clave:
    grafico_hist(dataset_personas_features[col], titulo, xlabel)

### 15.2 Nivel academico maximo (feature derivada)

In [ ]:
grafico_barras_categoria(
    dataset_personas_features["NIVEL_ACADEMICO_MAXIMO"],
    "Personas por nivel academico maximo", orden=orden_nivel,
)

### 15.3 Cobertura de datos: variables con mas nulos

In [ ]:
top_nulos = pct_nulos_final[pct_nulos_final > 0].head(15)
grafico_barh_valor(
    top_nulos.index.tolist(), top_nulos.values.tolist(),
    "Top-15 features con mas nulos en el dataset final", xlabel="% de personas",
)

### 15.4 Asimetria: variables mas sesgadas (candidatas a log1p en 04)

In [ ]:
top_sesgo = feature_diagnostics.reindex(
    feature_diagnostics["SKEW"].abs().sort_values(ascending=False).index
).head(15)
grafico_barh_valor(
    top_sesgo["FEATURE"].tolist(), top_sesgo["SKEW"].tolist(),
    "Top-15 features por asimetria (|skew|)", xlabel="Skewness", color=COLOR_ORANGE, fmt="{:.1f}",
)

### 15.5 Correlacion entre indicadores representativos

In [ ]:
columnas_correlacion = [
    "ANTIGUEDAD_EFECTIVA_ANIOS", "N_CONTRATOS_TOTAL", "NUM_TITULACIONES", "NUM_CAPACITACIONES",
    "NUM_CURSOS", "TOTAL_HORAS_DOCENCIA", "PROMEDIO_HETEROEVALUACION", "NUM_IDIOMAS",
    "TOTAL_HORAS_POLITECNICAS", "NUM_PUBLICACIONES", "NUM_PROYECTOS_INVESTIGACION",
    "NUM_PROYECTOS_VINCULACION", "NUM_RECONOCIMIENTOS", "NUM_PUBLICACIONES_POR_ANIO",
]
corr_repr = dataset_personas_features[columnas_correlacion].corr()

cmap_diverg = LinearSegmentedColormap.from_list("blue_gray_red", ["#e34948", "#f0efec", "#2a78d6"])
fig, ax = plt.subplots(figsize=(9, 8))
im = ax.imshow(corr_repr, cmap=cmap_diverg, vmin=-1, vmax=1)
ax.set_xticks(range(len(columnas_correlacion))); ax.set_xticklabels(columnas_correlacion, rotation=45, ha="right")
ax.set_yticks(range(len(columnas_correlacion))); ax.set_yticklabels(columnas_correlacion)
for i in range(len(columnas_correlacion)):
    for j in range(len(columnas_correlacion)):
        valor = corr_repr.iloc[i, j]
        color_texto = "#ffffff" if abs(valor) > 0.6 else INK
        ax.text(j, i, f"{valor:.2f}", ha="center", va="center", fontsize=7, color=color_texto)
ax.set_title("Correlacion entre indicadores representativos del dataset final", loc="left", fontsize=12, pad=12)
fig.colorbar(im, ax=ax, shrink=0.8, label="Correlacion de Pearson")
plt.tight_layout()
plt.show()